# CGAP on WESAD — Complete SHAP Explainability Notebook

This notebook provides **global and local feature explanations** for the CGAP model used in the WESAD project:
- Global branch: median imputation + `ExtraTreesClassifier`
- Online branch: median imputation + `StandardScaler` + `SGDClassifier(loss="log_loss")`
- Probability fusion: `(1 - lambda) * global_probability + lambda * online_probability`

It reads the corrected participant window files (`S1_windows.csv` … `S15_windows.csv`) used by the CGAP LOSO workflow. It fits one **illustrative outer fold** (one held-out participant) for explanation. This is not a replacement for the full 15-participant evaluation, and the explanations should be regenerated for multiple folds if you want participant-robust interpretation.

**Important interpretation notes**
1. The code explains the initial fitted branch models. The online SGD model changes after `partial_fit`; explanations here do not automatically describe every later online state.
2. The combined CGAP explanation is a probability-level weighted combination of branch explanations, valid for the selected class output when both branch SHAP values use the same feature columns and compatible probability output scale.
3. SHAP explains model behavior, not causal effects or physiological mechanisms.
4. Choose `LAMBDA_ONLINE` from the selected hyperparameter row for the held-out participant/seed if available. A default of `0.50` is used only when you do not supply the selected value.


## 1. Install packages and mount Google Drive

In [ ]:
%pip -q install shap scikit-learn pandas numpy matplotlib seaborn

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import re, warnings, json, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import classification_report, accuracy_score, f1_score

SEED = 42
HELD_OUT_SUBJECT = "S9"   # Change to any subject present in the CSVs, e.g. "S1"
LAMBDA_ONLINE = 0.50      # Replace with the selected fusion weight for this fold if known
MAX_BACKGROUND = 100      # SHAP background rows; increase if runtime allows
MAX_EXPLAIN = 300         # Rows explained in summary plots
OUTPUT_DIR = Path("/content/drive/MyDrive/DataCon/Project 6 Wesad/cgap_shap_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR = Path("/content/drive/MyDrive/DataCon/Project 6 Wesad")
print("SHAP version:", shap.__version__)
print("Data directory:", DATA_DIR)
print("Output directory:", OUTPUT_DIR)


## 2. Load and validate the same participant-level modelling data

In [ ]:
subject_files = sorted(
    p for p in DATA_DIR.glob("S*_windows.csv")
    if re.fullmatch(r"S\d+_windows", p.stem)
)
if len(subject_files) != 15:
    raise FileNotFoundError(
        f"Expected 15 files named S1_windows.csv ... S15_windows.csv in {DATA_DIR}; "
        f"found {len(subject_files)}. Update DATA_DIR or check filenames."
    )

frames = []
for path in subject_files:
    part = pd.read_csv(path)
    if "subject" not in part.columns:
        # Use filename only if the file has no subject metadata.
        part["subject"] = path.stem.split("_")[0]
    frames.append(part)

df = pd.concat(frames, ignore_index=True)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Subjects:", sorted(df["subject"].astype(str).unique().tolist()))
display(df.head())


## 3. Identify metadata, labels, and feature columns

In [ ]:
# Adjust these names only if your CSV schema uses different metadata names.
LABEL_CANDIDATES = ["label", "state", "target", "class", "y"]
SUBJECT_CANDIDATES = ["subject", "participant", "participant_id", "user_id"]
ORDER_CANDIDATES = ["window_idx", "window_index", "window", "timestamp", "start_time"]

def first_existing(candidates, columns):
    return next((c for c in candidates if c in columns), None)

label_col = first_existing(LABEL_CANDIDATES, df.columns)
subject_col = first_existing(SUBJECT_CANDIDATES, df.columns)
order_col = first_existing(ORDER_CANDIDATES, df.columns)

if label_col is None:
    raise ValueError(f"Could not identify label column. Tried {LABEL_CANDIDATES}.")
if subject_col is None:
    raise ValueError(f"Could not identify participant column. Tried {SUBJECT_CANDIDATES}.")

metadata_cols = {label_col, subject_col}
if order_col:
    metadata_cols.add(order_col)
# Common metadata that must not enter the feature matrix
metadata_cols.update(c for c in ["subject", "participant", "participant_id", "user_id",
                                  "label", "state", "target", "class", "y",
                                  "window_idx", "window_index", "window", "timestamp",
                                  "start_time", "end_time", "source_file"] if c in df.columns)

feature_cols = [
    c for c in df.columns
    if c not in metadata_cols and pd.api.types.is_numeric_dtype(df[c])
]
if not feature_cols:
    raise ValueError("No numeric feature columns found after excluding metadata.")

df[subject_col] = df[subject_col].astype(str)
df = df.sort_values([subject_col, order_col] if order_col else [subject_col]).reset_index(drop=True)

# This workflow expects clean precomputed features, consistent with the CGAP LOSO notebook.
X_all = df[feature_cols].replace([np.inf, -np.inf], np.nan)
y_text = df[label_col].astype(str)
if X_all.isna().any().any():
    missing = X_all.isna().sum().sort_values(ascending=False)
    print("Missing values detected; branch imputers will handle them. Top columns:")
    display(missing[missing > 0].head(20).to_frame("missing_count"))

print("Label column:", label_col)
print("Subject column:", subject_col)
print("Order column:", order_col)
print("Number of features:", len(feature_cols))
print("Features:", feature_cols)
print("Class counts:")
display(y_text.value_counts().to_frame("windows"))


## 4. Create one participant-held-out fold and fit the CGAP branches

In [ ]:
subjects = sorted(df[subject_col].unique().tolist())
if HELD_OUT_SUBJECT not in subjects:
    raise ValueError(f"{HELD_OUT_SUBJECT} not found. Available subjects: {subjects}")

train_mask = df[subject_col] != HELD_OUT_SUBJECT
test_mask = df[subject_col] == HELD_OUT_SUBJECT

X_train_raw = X_all.loc[train_mask, feature_cols].copy()
X_test_raw = X_all.loc[test_mask, feature_cols].copy()
y_train_text = y_text.loc[train_mask].copy()
y_test_text = y_text.loc[test_mask].copy()

# Encode labels once using training labels; fail if held-out contains an unseen class.
label_encoder = LabelEncoder().fit(y_train_text)
y_train = label_encoder.transform(y_train_text)
unknown_test = sorted(set(y_test_text) - set(label_encoder.classes_))
if unknown_test:
    raise ValueError(f"Unseen test labels: {unknown_test}")
y_test = label_encoder.transform(y_test_text)
classes = label_encoder.classes_
n_classes = len(classes)

if n_classes < 2:
    raise ValueError("Training data has fewer than two classes.")

# Global branch: median imputer + ExtraTrees.
global_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", ExtraTreesClassifier(
        n_estimators=400,
        max_features="sqrt",
        class_weight="balanced",
        random_state=SEED,
        n_jobs=-1
    ))
])
global_model.fit(X_train_raw, y_train)

# Online branch: median imputer + scaler + logistic-loss SGD.
online_imputer = SimpleImputer(strategy="median")
online_scaler = StandardScaler()
X_train_imp = online_imputer.fit_transform(X_train_raw)
X_train_scaled = online_scaler.fit_transform(X_train_imp)
X_test_imp = online_imputer.transform(X_test_raw)
X_test_scaled = online_scaler.transform(X_test_imp)

online_model = SGDClassifier(
    loss="log_loss",
    alpha=0.001,
    max_iter=1500,
    tol=1e-3,
    random_state=SEED,
    class_weight=None
)
sample_weights = compute_sample_weight(class_weight="balanced", y=y_train)
online_model.fit(X_train_scaled, y_train, sample_weight=sample_weights)

# Align model probability columns to the encoded class order.
def aligned_predict_proba(model, X, all_classes):
    raw = model.predict_proba(X)
    model_classes = model.classes_
    aligned = np.zeros((len(X), len(all_classes)), dtype=float)
    for j, cls in enumerate(model_classes):
        aligned[:, int(cls)] = raw[:, j]
    return aligned

X_test_global = global_model.named_steps["imputer"].transform(X_test_raw)
global_probs = aligned_predict_proba(global_model.named_steps["model"], X_test_global, np.arange(n_classes))
online_probs = aligned_predict_proba(online_model, X_test_scaled, np.arange(n_classes))
cgap_probs = (1.0 - LAMBDA_ONLINE) * global_probs + LAMBDA_ONLINE * online_probs

for name, probs in [("Global only", global_probs), ("Online only", online_probs), ("CGAP fused", cgap_probs)]:
    pred = probs.argmax(axis=1)
    print(f"\n{name}: accuracy={accuracy_score(y_test, pred):.4f}, "
          f"macro-F1={f1_score(y_test, pred, average='macro', zero_division=0):.4f}")
print("\nCGAP fused classification report:")
print(classification_report(y_test, cgap_probs.argmax(axis=1),
                            labels=np.arange(n_classes), target_names=classes,
                            zero_division=0))
print("Training rows:", len(X_train_raw), "| held-out rows:", len(X_test_raw))
print("Fusion weight lambda:", LAMBDA_ONLINE)


## 5. Prepare SHAP background and explanation samples

In [ ]:
rng = np.random.default_rng(SEED)
n_bg = min(MAX_BACKGROUND, len(X_train_raw))
n_explain = min(MAX_EXPLAIN, len(X_test_raw))

bg_idx = rng.choice(len(X_train_raw), size=n_bg, replace=False)
explain_idx = np.sort(rng.choice(len(X_test_raw), size=n_explain, replace=False))

X_bg_raw = X_train_raw.iloc[bg_idx].copy()
X_explain_raw = X_test_raw.iloc[explain_idx].copy()
X_explain_global = global_model.named_steps["imputer"].transform(X_explain_raw)
X_bg_global = global_model.named_steps["imputer"].transform(X_bg_raw)

X_bg_online = online_scaler.transform(online_imputer.transform(X_bg_raw))
X_explain_online = online_scaler.transform(online_imputer.transform(X_explain_raw))

print("Background rows:", len(X_bg_raw))
print("Rows to explain:", len(X_explain_raw))


## 6. SHAP for the global ExtraTrees branch

In [ ]:
tree_explainer = shap.TreeExplainer(
    global_model.named_steps["model"],
    data=X_bg_global,
    feature_perturbation="interventional",
    model_output="probability"
)
tree_shap_raw = tree_explainer.shap_values(X_explain_global)

def shap_for_class(shap_values, class_idx, n_features):
    # SHAP API differs by version: list[class] or ndarray (samples, features, classes).
    if isinstance(shap_values, list):
        return np.asarray(shap_values[class_idx])
    arr = np.asarray(shap_values)
    if arr.ndim == 3:
        # Common layout: samples x features x classes
        if arr.shape[1] == n_features:
            return arr[:, :, class_idx]
        # Alternate layout: classes x samples x features
        if arr.shape[0] > class_idx and arr.shape[2] == n_features:
            return arr[class_idx]
    if arr.ndim == 2:
        return arr
    raise ValueError(f"Unrecognized SHAP output shape: {arr.shape}")

print("Tree SHAP output type:", type(tree_shap_raw))
if isinstance(tree_shap_raw, list):
    print("Per-class shapes:", [np.asarray(v).shape for v in tree_shap_raw])
else:
    print("SHAP shape:", np.asarray(tree_shap_raw).shape)


## 7. Global feature importance and class-specific summary plots

In [ ]:
# Global importance: mean absolute SHAP value, averaged over classes.
tree_class_shaps = [
    shap_for_class(tree_shap_raw, k, len(feature_cols))
    for k in range(n_classes)
]
tree_abs_mean = np.mean(
    np.stack([np.abs(s).mean(axis=0) for s in tree_class_shaps], axis=0),
    axis=0
)
tree_importance = pd.DataFrame({
    "feature": feature_cols,
    "mean_abs_shap_global_branch": tree_abs_mean
}).sort_values("mean_abs_shap_global_branch", ascending=False)

display(tree_importance.head(20))
tree_importance.to_csv(OUTPUT_DIR / f"shap_global_importance_{HELD_OUT_SUBJECT}.csv", index=False)

plt.figure(figsize=(9, 7))
sns.barplot(data=tree_importance.head(20), x="mean_abs_shap_global_branch", y="feature")
plt.title(f"Global branch: mean |SHAP| across classes ({HELD_OUT_SUBJECT} held out)")
plt.xlabel("Mean absolute SHAP value")
plt.ylabel("Feature")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / f"shap_global_bar_{HELD_OUT_SUBJECT}.png", dpi=300, bbox_inches="tight")
plt.show()

for k, class_name in enumerate(classes):
    print(f"Global branch SHAP summary — class: {class_name}")
    shap.summary_plot(
        tree_class_shaps[k],
        X_explain_global,
        feature_names=feature_cols,
        max_display=20,
        show=True
    )


## 8. SHAP for the online SGD branch

In [ ]:
# SGDClassifier(loss='log_loss') is a linear probabilistic classifier.
# LinearExplainer is applied in the transformed feature space used by SGD.
linear_explainer = shap.LinearExplainer(
    online_model,
    X_bg_online,
    feature_perturbation="interventional"
)
linear_shap_raw = linear_explainer.shap_values(X_explain_online)

linear_class_shaps = [
    shap_for_class(linear_shap_raw, k, len(feature_cols))
    for k in range(n_classes)
]
linear_abs_mean = np.mean(
    np.stack([np.abs(s).mean(axis=0) for s in linear_class_shaps], axis=0),
    axis=0
)
linear_importance = pd.DataFrame({
    "feature": feature_cols,
    "mean_abs_shap_online_branch": linear_abs_mean
}).sort_values("mean_abs_shap_online_branch", ascending=False)

display(linear_importance.head(20))
linear_importance.to_csv(OUTPUT_DIR / f"shap_online_importance_{HELD_OUT_SUBJECT}.csv", index=False)

plt.figure(figsize=(9, 7))
sns.barplot(data=linear_importance.head(20), x="mean_abs_shap_online_branch", y="feature")
plt.title(f"Online SGD branch: mean |SHAP| across classes ({HELD_OUT_SUBJECT} held out)")
plt.xlabel("Mean absolute SHAP value in SGD's transformed feature space")
plt.ylabel("Feature")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / f"shap_online_bar_{HELD_OUT_SUBJECT}.png", dpi=300, bbox_inches="tight")
plt.show()

for k, class_name in enumerate(classes):
    print(f"Online branch SHAP summary — class: {class_name}")
    shap.summary_plot(
        linear_class_shaps[k],
        X_explain_online,
        feature_names=feature_cols,
        max_display=20,
        show=True
    )


## 9. Probability-level combined CGAP SHAP values

For a selected class, the fused probability is a weighted sum of the two branch probabilities. Because SHAP values are additive for the corresponding probability outputs, the branch SHAP values can be combined using the same fusion weights **when the explainers' outputs are both on probability scale and use the same feature representation**. The code below first checks the expected-value and SHAP shapes and constructs a class-wise weighted combination. If your SHAP version/model-output behavior differs, treat branch-level explanations as the primary result and do not report a combined SHAP plot until the additivity check passes.

In [ ]:
# Confirm the TreeExplainer's probability output and linear explainer output shapes before fusion.
tree_base_raw = tree_explainer.expected_value
linear_base_raw = linear_explainer.expected_value
print("Tree expected_value:", np.asarray(tree_base_raw).shape, tree_base_raw)
print("Linear expected_value:", np.asarray(linear_base_raw).shape, linear_base_raw)

def base_for_class(base_values, class_idx, n_classes):
    arr = np.asarray(base_values)
    if arr.ndim == 0:
        return float(arr)
    arr = arr.reshape(-1)
    if len(arr) == n_classes:
        return float(arr[class_idx])
    if len(arr) == 1:
        return float(arr[0])
    raise ValueError(f"Cannot map expected_value shape {arr.shape} to {n_classes} classes.")

combined_rows = []
for k, class_name in enumerate(classes):
    tree_s = tree_class_shaps[k]
    linear_s = linear_class_shaps[k]
    combined_s = (1.0 - LAMBDA_ONLINE) * tree_s + LAMBDA_ONLINE * linear_s
    tree_base_k = base_for_class(tree_base_raw, k, n_classes)
    linear_base_k = base_for_class(linear_base_raw, k, n_classes)
    combined_base_k = (1.0 - LAMBDA_ONLINE) * tree_base_k + LAMBDA_ONLINE * linear_base_k

    # Fused probability values for this class.
    target_probs = cgap_probs[explain_idx, k]
    reconstructed = combined_base_k + combined_s.sum(axis=1)
    mae = float(np.mean(np.abs(target_probs - reconstructed)))
    max_abs_err = float(np.max(np.abs(target_probs - reconstructed)))
    print(f"{class_name}: additivity MAE={mae:.6f}, max abs error={max_abs_err:.6f}")

    combined_rows.append({
        "class": class_name,
        "additivity_mae": mae,
        "additivity_max_abs_error": max_abs_err
    })

pd.DataFrame(combined_rows).to_csv(
    OUTPUT_DIR / f"shap_fusion_additivity_check_{HELD_OUT_SUBJECT}.csv", index=False
)


In [ ]:
# Plot combined explanations only if the numerical additivity check is sufficiently close.
ADDIVITY_TOLERANCE = 0.05
check_df = pd.DataFrame(combined_rows)
if check_df["additivity_mae"].max() <= ADDIVITY_TOLERANCE:
    combined_importance = np.mean(
        np.stack([
            np.abs((1.0 - LAMBDA_ONLINE) * tree_class_shaps[k]
                   + LAMBDA_ONLINE * linear_class_shaps[k]).mean(axis=0)
            for k in range(n_classes)
        ], axis=0),
        axis=0
    )
    combined_importance_df = pd.DataFrame({
        "feature": feature_cols,
        "mean_abs_shap_cgap_fused": combined_importance
    }).sort_values("mean_abs_shap_cgap_fused", ascending=False)
    display(combined_importance_df.head(20))
    combined_importance_df.to_csv(
        OUTPUT_DIR / f"shap_cgap_fused_importance_{HELD_OUT_SUBJECT}.csv", index=False
    )

    plt.figure(figsize=(9, 7))
    sns.barplot(data=combined_importance_df.head(20),
                x="mean_abs_shap_cgap_fused", y="feature")
    plt.title(f"CGAP fused probability: mean |SHAP| across classes ({HELD_OUT_SUBJECT})")
    plt.xlabel("Mean absolute fused SHAP value")
    plt.ylabel("Feature")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"shap_cgap_fused_bar_{HELD_OUT_SUBJECT}.png",
                dpi=300, bbox_inches="tight")
    plt.show()

    # Class-specific combined beeswarm plots.
    for k, class_name in enumerate(classes):
        combined_s = (1.0 - LAMBDA_ONLINE) * tree_class_shaps[k] + LAMBDA_ONLINE * linear_class_shaps[k]
        print(f"CGAP fused SHAP summary — class: {class_name}")
        shap.summary_plot(
            combined_s,
            X_explain_raw.to_numpy(),
            feature_names=feature_cols,
            max_display=20,
            show=True
        )
else:
    print(
        "Combined SHAP plot skipped: additivity error is above tolerance. "
        "Use global and online branch explanations separately; inspect SHAP output shapes/API."
    )


## 10. Local explanation for one held-out window

In [ ]:
# Choose the class with the highest fused probability for the first explained window.
local_row = 0
local_class = int(cgap_probs[explain_idx[local_row]].argmax())
local_class_name = classes[local_class]
print("Original held-out row index:", int(explain_idx[local_row]))
print("True label:", y_test_text.iloc[explain_idx[local_row]])
print("Predicted class:", local_class_name)
print("Fused probabilities:", dict(zip(classes, cgap_probs[explain_idx[local_row]].round(4))))

# Waterfall plot for the global branch.
tree_exp = shap.Explanation(
    values=tree_class_shaps[local_class][local_row],
    base_values=base_for_class(tree_base_raw, local_class, n_classes),
    data=X_explain_global[local_row],
    feature_names=feature_cols
)
shap.plots.waterfall(tree_exp, max_display=15, show=True)

# Waterfall plot for online branch (values are in scaled-feature model space).
linear_exp = shap.Explanation(
    values=linear_class_shaps[local_class][local_row],
    base_values=base_for_class(linear_base_raw, local_class, n_classes),
    data=X_explain_online[local_row],
    feature_names=feature_cols
)
shap.plots.waterfall(linear_exp, max_display=15, show=True)

# If additivity check passed, local fused explanation is also available.
if "check_df" in globals() and check_df["additivity_mae"].max() <= ADDIVITY_TOLERANCE:
    local_combined_values = (
        (1.0 - LAMBDA_ONLINE) * tree_class_shaps[local_class][local_row]
        + LAMBDA_ONLINE * linear_class_shaps[local_class][local_row]
    )
    local_combined_base = (
        (1.0 - LAMBDA_ONLINE) * base_for_class(tree_base_raw, local_class, n_classes)
        + LAMBDA_ONLINE * base_for_class(linear_base_raw, local_class, n_classes)
    )
    combined_exp = shap.Explanation(
        values=local_combined_values,
        base_values=local_combined_base,
        data=X_explain_raw.iloc[local_row].to_numpy(),
        feature_names=feature_cols
    )
    shap.plots.waterfall(combined_exp, max_display=15, show=True)


## 11. Export reproducible explanation tables and metadata

In [ ]:
metadata = {
    "held_out_subject": HELD_OUT_SUBJECT,
    "seed": SEED,
    "lambda_online": LAMBDA_ONLINE,
    "label_column": label_col,
    "subject_column": subject_col,
    "order_column": order_col,
    "n_train_rows": int(len(X_train_raw)),
    "n_test_rows": int(len(X_test_raw)),
    "n_explained_rows": int(len(X_explain_raw)),
    "features": feature_cols,
    "classes": classes.tolist(),
    "global_branch": "SimpleImputer(median) + ExtraTreesClassifier",
    "online_branch": "SimpleImputer(median) + StandardScaler + SGDClassifier(loss=log_loss)",
    "interpretation_scope": "initial fitted models; not per-time-step post-update explanations",
    "shap_version": shap.__version__
}
with open(OUTPUT_DIR / f"shap_run_metadata_{HELD_OUT_SUBJECT}.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Saved SHAP outputs to:", OUTPUT_DIR)
for p in sorted(OUTPUT_DIR.glob(f"*{HELD_OUT_SUBJECT}*")):
    print(p.name)


## 12. Reporting checklist for a paper

- Report the held-out participant(s), training/background sample size, SHAP version, and feature set.
- Use **mean absolute SHAP values** as model-attribution importance, not as evidence of causal importance.
- Provide class-specific summaries because baseline, stress, and amusement may rely on different features.
- State that the ExtraTrees SHAP values explain the global branch, while Linear SHAP explains the initial online SGD branch.
- If using fused CGAP explanations, report the numerical additivity check and the chosen fusion weight.
- For robust claims, repeat this analysis across all 15 outer LOSO folds and aggregate feature attributions by participant; do not treat windows as independent participants.
- Because test labels are used for sequential `partial_fit` updates in the evaluation protocol, this notebook's static explanations describe the **initial fitted models**, not each post-feedback update state.
